# Run K: SegFormer-B0, binary defect segmentation

SegFormer was published at NeurIPS 2021: *SegFormer: Simple and Efficient Design for Semantic Segmentation with Transformers*. This notebook uses the binary masks from `SmallDefectPreprocessing`, recreates the fixed size-stratified split used in the YOLO runs, early-stops on validation Dice, and evaluates overall/small/medium/large test subsets.

Kaggle setup: attach **SmallDefectPreprocessing** as an input, enable Internet for the pretrained SegFormer-B0 weights, and use a GPU.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import copy
import json
import os
import random
import shutil
import subprocess
import sys
import time

RUN_NAME = 'RunK_segformer_b0_imgsz640'
MODEL_NAME = 'nvidia/mit-b0'
MODEL_LABEL = 'SegFormer-B0'
WANDB_PROJECT = 'smallDefectDetection'
WANDB_RUN_NAME = f'{MODEL_LABEL}_segmentation'
IMG_SIZE = 640
BATCH_SIZE = 8
MAX_EPOCHS = 50
PATIENCE = 10
LEARNING_RATE = 6e-5
WEIGHT_DECAY = 1e-2
NUM_WORKERS = 2
SEED = 42

DATASET_NAMES = ['DAGM', 'GC10-DET', 'KolektorSDD2', 'MPDD', 'MTD', 'Severstal', 'VisA']
SIZE_BUCKETS = ['small', 'medium', 'large']
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

KAGGLE_INPUT_ROOT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working')
RUN_DIR = WORKING_ROOT / 'segformer_runs' / RUN_NAME
FINAL_OUTPUT_DIR = WORKING_ROOT / 'final_outputs' / RUN_NAME

print({'run': RUN_NAME, 'model': MODEL_NAME, 'imgsz': IMG_SIZE, 'batch': BATCH_SIZE, 'max_epochs': MAX_EPOCHS, 'patience': PATIENCE})

In [ ]:
# Kaggle Internet must be enabled for this cell.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.45.0', 'safetensors', 'wandb'], check=True)

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import wandb
from PIL import Image
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('Enable a Kaggle GPU before training.')

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.benchmark = True
print('Using device:', torch.cuda.get_device_name(0))


def wandb_login_anywhere():
    # Works on RunPod (env var) and Kaggle (Secrets add-on) without ever hardcoding the key.
    api_key = os.environ.get('WANDB_API_KEY')
    if not api_key:
        try:
            from kaggle_secrets import UserSecretsClient
            api_key = UserSecretsClient().get_secret('WANDB_API_KEY')
        except Exception:
            api_key = None
    if api_key:
        wandb.login(key=api_key)
    else:
        wandb.login()


wandb_login_anywhere()
wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        'model': MODEL_NAME,
        'img_size': IMG_SIZE,
        'batch_size': BATCH_SIZE,
        'max_epochs': MAX_EPOCHS,
        'patience': PATIENCE,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'seed': SEED,
    },
)

In [ ]:
# Find the normal SmallDefectPreprocessing Kaggle input automatically.
expected_datasets = set(DATASET_NAMES)
source_candidates = []

for root, dirs, _ in os.walk(KAGGLE_INPUT_ROOT):
    matches = expected_datasets.intersection(dirs)
    if len(matches) >= 5:
        source_candidates.append((len(matches), Path(root)))

if not source_candidates:
    raise FileNotFoundError(
        'Could not find the processed dataset. Attach SmallDefectPreprocessing as a Kaggle input.'
    )

source_candidates.sort(key=lambda item: (-item[0], len(str(item[1]))))
SOURCE_ROOT = source_candidates[0][1]
print('Using processed source:', SOURCE_ROOT)
print('Datasets:', sorted(path.name for path in SOURCE_ROOT.iterdir() if path.is_dir()))

In [ ]:
def index_files(directory, suffixes):
    return {
        path.stem: path
        for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() in suffixes
    }

def candidate_stems(image_stem, target):
    base = image_stem.removesuffix('_defect')
    if target == 'mask':
        return [image_stem, image_stem.replace('_defect', '_mask'), base, base + '_mask', base + '_gt']
    return [image_stem, image_stem.replace('_defect', '_bbs'), base, base + '_bbs']

samples = []
missing = []

for dataset_name in DATASET_NAMES:
    for size_bucket in SIZE_BUCKETS:
        bucket_root = SOURCE_ROOT / dataset_name / size_bucket
        image_dir = bucket_root / 'images'
        mask_dir = bucket_root / 'masks'
        label_dir = bucket_root / 'labels_yolo'

        if not image_dir.exists() or not mask_dir.exists() or not label_dir.exists():
            missing.append((dataset_name, size_bucket, 'missing directory'))
            continue

        mask_index = index_files(mask_dir, IMAGE_EXTS)
        label_index = index_files(label_dir, {'.txt'})
        matched = 0

        for image_path in image_dir.iterdir():
            if image_path.suffix.lower() not in IMAGE_EXTS:
                continue

            mask_path = next((mask_index[stem] for stem in candidate_stems(image_path.stem, 'mask') if stem in mask_index), None)
            label_path = next((label_index[stem] for stem in candidate_stems(image_path.stem, 'box') if stem in label_index), None)

            if mask_path is None or label_path is None:
                missing.append((dataset_name, size_bucket, image_path.name))
                continue

            samples.append({
                'image_path': image_path,
                'mask_path': mask_path,
                'label_path': label_path,
                'dataset': dataset_name,
                'size': size_bucket,
                'stratum': dataset_name + '_' + size_bucket,
            })
            matched += 1

        print(f'{dataset_name}/{size_bucket}: {matched} matched')

print('Total matched:', len(samples))
print('Missing:', len(missing))
if len(samples) != 12670:
    raise RuntimeError(f'Expected 12,670 image-mask-label triplets, found {len(samples)}. First missing: {missing[:10]}')

In [ ]:
# Same fixed 70/15/15 stratified split as the YOLO experiments.
by_stratum = defaultdict(list)
for sample in samples:
    by_stratum[sample['stratum']].append(sample)

rng = random.Random(SEED)
train_samples, val_samples, test_samples = [], [], []
for _, group in sorted(by_stratum.items()):
    group = list(group)
    rng.shuffle(group)
    train_end = int(len(group) * 0.70)
    val_end = train_end + int(len(group) * 0.15)
    train_samples.extend(group[:train_end])
    val_samples.extend(group[train_end:val_end])
    test_samples.extend(group[val_end:])

rng.shuffle(train_samples)
rng.shuffle(val_samples)
rng.shuffle(test_samples)

test_sets = {
    'overall': test_samples,
    'small': [sample for sample in test_samples if sample['size'] == 'small'],
    'medium': [sample for sample in test_samples if sample['size'] == 'medium'],
    'large': [sample for sample in test_samples if sample['size'] == 'large'],
}

def print_counts(name, split):
    counts = Counter(sample['size'] for sample in split)
    print(f'{name}: total={len(split)}, small={counts["small"]}, medium={counts["medium"]}, large={counts["large"]}')

print_counts('Train', train_samples)
print_counts('Validation', val_samples)
for name, split in test_sets.items():
    print_counts('Test ' + name, split)

assert len(train_samples) == 8858
assert len(val_samples) == 1892
assert len(test_samples) == 1920

In [ ]:
def load_binary_mask(mask_path, target_size):
    mask = Image.open(mask_path).convert('L')
    if mask.size != target_size:
        mask = mask.resize(target_size, Image.Resampling.NEAREST)
    return (np.asarray(mask) > 0).astype(np.uint8)

processor = SegformerImageProcessor(
    do_resize=True,
    size={'height': IMG_SIZE, 'width': IMG_SIZE},
    do_reduce_labels=False,
)

class DefectMaskDataset(Dataset):
    def __init__(self, split_samples):
        self.samples = split_samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        image = Image.open(sample['image_path']).convert('RGB')
        mask = load_binary_mask(sample['mask_path'], image.size)
        encoded = processor(images=image, segmentation_maps=mask, return_tensors='pt')
        return {
            'pixel_values': encoded['pixel_values'].squeeze(0),
            'labels': encoded['labels'].squeeze(0).long(),
        }

def make_loader(split_samples, shuffle=False):
    return DataLoader(
        DefectMaskDataset(split_samples),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0,
    )

train_loader = make_loader(train_samples, shuffle=True)
val_loader = make_loader(val_samples)
print('Train batches:', len(train_loader), 'Validation batches:', len(val_loader))

In [ ]:
model = SegformerForSemanticSegmentation.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'background', 1: 'defect'},
    label2id={'background': 0, 'defect': 1},
    ignore_mismatched_sizes=True,
).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler('cuda', enabled=True)

RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Model loaded. Trainable parameters:', sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad))

In [ ]:
@torch.inference_mode()
def evaluate(loader, measure_inference=False):
    model.eval()
    true_positive = false_positive = false_negative = 0
    inference_seconds = 0.0
    image_count = 0

    for batch in loader:
        pixel_values = batch['pixel_values'].to(DEVICE, non_blocking=True)
        labels = batch['labels'].to(DEVICE, non_blocking=True)

        if measure_inference:
            torch.cuda.synchronize()
            start = time.perf_counter()
        outputs = model(pixel_values=pixel_values)
        if measure_inference:
            torch.cuda.synchronize()
            inference_seconds += time.perf_counter() - start

        logits = F.interpolate(outputs.logits, size=labels.shape[-2:], mode='bilinear', align_corners=False)
        predictions = logits.argmax(dim=1)
        true_positive += int(((predictions == 1) & (labels == 1)).sum().item())
        false_positive += int(((predictions == 1) & (labels == 0)).sum().item())
        false_negative += int(((predictions == 0) & (labels == 1)).sum().item())
        image_count += labels.shape[0]

    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)
    iou = true_positive / max(true_positive + false_positive + false_negative, 1)
    dice = 2 * true_positive / max(2 * true_positive + false_positive + false_negative, 1)

    return {
        'precision': precision,
        'recall': recall,
        'iou': iou,
        'dice': dice,
        'inference_time_ms_per_image': 1000 * inference_seconds / max(image_count, 1),
        'images': image_count,
    }

history = []
best_dice = -1.0
best_epoch = -1
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for batch in train_loader:
        pixel_values = batch['pixel_values'].to(DEVICE, non_blocking=True)
        labels = batch['labels'].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += float(loss.item())

    val_metrics = evaluate(val_loader)
    row = {'epoch': epoch, 'train_loss': running_loss / len(train_loader), **val_metrics}
    history.append(row)
    print(f"Epoch {epoch:02d}/{MAX_EPOCHS} | loss={row['train_loss']:.4f} | val Dice={row['dice']:.4f} | val IoU={row['iou']:.4f} | val Recall={row['recall']:.4f}")
    wandb.log(row, step=epoch)

    if row['dice'] > best_dice:
        best_dice = row['dice']
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_metrics': val_metrics,
        }, RUN_DIR / 'best_model.pt')
    else:
        epochs_without_improvement += 1

    pd.DataFrame(history).to_csv(RUN_DIR / 'training_history.csv', index=False)
    if epochs_without_improvement >= PATIENCE:
        print(f'Early stopping at epoch {epoch}; best validation Dice was {best_dice:.4f} at epoch {best_epoch}.')
        break

wandb.summary['best_epoch'] = best_epoch
wandb.summary['best_validation_dice'] = best_dice
print('Best epoch:', best_epoch, 'Best validation Dice:', best_dice)

In [ ]:
# Load the best validation-Dice checkpoint and evaluate every fixed test subset.
checkpoint = torch.load(RUN_DIR / 'best_model.pt', map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

test_rows = []
for split_name, split_samples in test_sets.items():
    metrics = evaluate(make_loader(split_samples), measure_inference=True)
    test_rows.append({'split': split_name, **metrics})
    print(split_name, metrics)
    wandb.log({f'test_{split_name}_{key}': value for key, value in metrics.items()})

test_df = pd.DataFrame(test_rows)
display(test_df)

FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(RUN_DIR / 'best_model.pt', FINAL_OUTPUT_DIR / 'best_model.pt')
shutil.copy2(RUN_DIR / 'training_history.csv', FINAL_OUTPUT_DIR / 'training_history.csv')
test_df.to_csv(FINAL_OUTPUT_DIR / 'test_metrics.csv', index=False)

metadata = {
    'experiment': RUN_NAME,
    'model': 'SegFormer-B0',
    'task': 'binary semantic defect segmentation',
    'image_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'max_epochs': MAX_EPOCHS,
    'early_stopping_patience': PATIENCE,
    'best_epoch': best_epoch,
    'best_validation_dice': best_dice,
    'split_counts': {
        'train': len(train_samples), 'val': len(val_samples),
        'test': len(test_samples), 'test_small': len(test_sets['small']),
        'test_medium': len(test_sets['medium']), 'test_large': len(test_sets['large']),
    },
}
(FINAL_OUTPUT_DIR / 'run_metadata.json').write_text(json.dumps(metadata, indent=2))
print('Saved model and all metrics to:', FINAL_OUTPUT_DIR)

wandb.finish()